# PySpark DataFrame Data Engineering Project — Agriculture Analytics

Business-oriented PySpark DataFrame project using two related input files.

**Input files:** `farms.csv`, `harvest_records.csv`

**Questions:** 30

Topics include DataFrame creation, schema, select, filter, withColumn, when, cast, groupBy, agg, joins, orderBy, window functions, null handling, duplicates, date functions, pivot, union-style reporting, and business KPIs.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.window import Window

spark = SparkSession.builder.appName("PySparkDataFrameProject").getOrCreate()

df1 = spark.read.option("header", True).option("inferSchema", True).csv("farms.csv")
df2 = spark.read.option("header", True).option("inferSchema", True).csv("harvest_records.csv")

df1.show(5, truncate=False)
df2.show(5, truncate=False)


## Q1. Inspect Schemas

Display both DataFrame schemas and row counts.

In [ ]:
df1.printSchema()
df2.printSchema()
print(df1.count(), df2.count())

## Q2. Select Business Columns

Select farm_id and district from the first DataFrame.

In [ ]:
df1.select('farm_id','district').show(20, truncate=False)

## Q3. Filter Records

Filter the second DataFrame where revenue is greater than its average.

In [ ]:
avg_value = df2.select(avg('revenue').alias('avg')).first()['avg']
df2.filter(col('revenue') > avg_value).show(20, truncate=False)

## Q4. Create Derived Column

Create a new column based on revenue using a High/Normal classification.

In [ ]:
df2.withColumn('value_category', when(col('revenue') >= avg('revenue').over(Window.partitionBy()), 'High').otherwise('Normal')).show(20, truncate=False)

## Q5. Null Analysis

Show null counts for every column in both DataFrames.

In [ ]:
df1.select([sum(col(c).isNull().cast('int')).alias(c) for c in df1.columns]).show()
df2.select([sum(col(c).isNull().cast('int')).alias(c) for c in df2.columns]).show()

## Q6. Handle Null Values

Fill null values in district and crop_type with 'Unknown'.

In [ ]:
df1.fillna({'district': 'Unknown'}).show(5)
df2.fillna({'crop_type': 'Unknown'}).show(5)

## Q7. Remove Duplicates

Remove duplicate records and compare row counts.

In [ ]:
print('Before:', df1.count(), df2.count())
print('After:', df1.dropDuplicates().count(), df2.dropDuplicates().count())

## Q8. Inner Join

Join the two DataFrames using farm_id and farm_id.

In [ ]:
joined = df1.join(df2, df1['farm_id'] == df2['farm_id'], 'inner')
joined.show(20, truncate=False)

## Q9. Left Join

Perform a left join and identify unmatched records.

In [ ]:
left = df1.join(df2, df1['farm_id'] == df2['farm_id'], 'left')
left.show(20, truncate=False)

## Q10. Group By Category

Calculate count and average revenue by crop_type.

In [ ]:
df2.groupBy('crop_type').agg(count('*').alias('records'), avg('revenue').alias('avg_value')).orderBy(desc('avg_value')).show()

## Q11. Total by Category

Calculate total revenue by crop_type.

In [ ]:
df2.groupBy('crop_type').agg(sum('revenue').alias('total_value')).orderBy(desc('total_value')).show()

## Q12. Maximum by Category

Find maximum revenue in each crop_type.

In [ ]:
df2.groupBy('crop_type').agg(max('revenue').alias('max_value')).show()

## Q13. Minimum by Category

Find minimum revenue in each crop_type.

In [ ]:
df2.groupBy('crop_type').agg(min('revenue').alias('min_value')).show()

## Q14. Having Condition

Show crop_type groups whose total revenue is greater than the overall average revenue.

In [ ]:
overall = df2.select(avg('revenue').alias('v')).first()['v']
df2.groupBy('crop_type').agg(sum('revenue').alias('total')).filter(col('total') > overall).show()

## Q15. Top 10 Records

Find the top 10 records by revenue.

In [ ]:
df2.orderBy(desc('revenue')).limit(10).show(truncate=False)

## Q16. Bottom 10 Records

Find the bottom 10 records by revenue.

In [ ]:
df2.orderBy(asc('revenue')).limit(10).show(truncate=False)

## Q17. Window Rank

Rank records by revenue within each crop_type.

In [ ]:
w = Window.partitionBy('crop_type').orderBy(desc('revenue'))
df2.withColumn('rank', dense_rank().over(w)).show(30, truncate=False)

## Q18. Running Total

Calculate a running total of revenue within each crop_type.

In [ ]:
w = Window.partitionBy('crop_type').orderBy('harvest_date').rowsBetween(Window.unboundedPreceding, Window.currentRow)
df2.withColumn('running_total', sum('revenue').over(w)).show(30, truncate=False)

## Q19. Previous Value

Use lag to compare each record's revenue with the previous record in its crop_type.

In [ ]:
w = Window.partitionBy('crop_type').orderBy('harvest_date')
df2.withColumn('previous_value', lag('revenue').over(w)).show(30, truncate=False)

## Q20. Percentage Contribution

Calculate each crop_type's contribution to total revenue.

In [ ]:
tot = df2.select(sum('revenue').alias('t')).first()['t']
df2.groupBy('crop_type').agg(sum('revenue').alias('total')).withColumn('pct', col('total')/lit(tot)*100).orderBy(desc('pct')).show()

## Q21. Date Extraction

Extract year and month from harvest_date.

In [ ]:
df2.withColumn('year', year('harvest_date')).withColumn('month', month('harvest_date')).show(20, truncate=False)

## Q22. Monthly KPI

Calculate total and average revenue by month.

In [ ]:
df2.withColumn('month', date_format('harvest_date','yyyy-MM')).groupBy('month').agg(sum('revenue').alias('total'), avg('revenue').alias('average')).orderBy('month').show()

## Q23. Pivot Report

Create a pivot report with crop_type as rows and year of harvest_date as columns.

In [ ]:
df2.withColumn('year', year('harvest_date')).groupBy('crop_type').pivot('year').agg(sum('revenue')).show()

## Q24. Conditional KPI

Count records where revenue is above the category average.

In [ ]:
w=Window.partitionBy('crop_type')
df2.withColumn('cat_avg',avg('revenue').over(w)).filter(col('revenue')>col('cat_avg')).groupBy('crop_type').count().show()

## Q25. Join Aggregation

Join the DataFrames and calculate total revenue by district.

In [ ]:
df1.join(df2, df1['farm_id']==df2['farm_id']).groupBy(df1['district']).agg(sum(df2['revenue']).alias('total')).orderBy(desc('total')).show()

## Q26. Join Ranking

After joining, rank district by total revenue.

In [ ]:
r=df1.join(df2,df1['farm_id']==df2['farm_id']).groupBy(df1['district']).agg(sum(df2['revenue']).alias('total'))
r.withColumn('rank',dense_rank().over(Window.orderBy(desc('total')))).show()

## Q27. Business Segmentation

Create Low, Medium and High segments for revenue.

In [ ]:
df2.withColumn('segment',when(col('revenue')<1000,'Low').when(col('revenue')<5000,'Medium').otherwise('High')).groupBy('segment').count().show()

## Q28. Data Quality Check

Count invalid negative revenue values.

In [ ]:
df2.filter(col('revenue')<0).count()

## Q29. Final Business Report

Create a final report by crop_type with count, total, average, minimum, maximum and rank.

In [ ]:
r=df2.groupBy('crop_type').agg(count('*').alias('records'),sum('revenue').alias('total'),avg('revenue').alias('average'),min('revenue').alias('minimum'),max('revenue').alias('maximum'))
r.withColumn('rank',dense_rank().over(Window.orderBy(desc('total')))).orderBy('rank').show(truncate=False)

## End of Project